# 05 · Modelado: modelos de la tesis sobre `training_dataset`

Generado por `PYTHONPATH=src python -m usab_ds stage5`. Este notebook solo **lee** los resultados.

- Partición **70/30 por participante** (ninguna persona en ambos conjuntos).
- Hiperparámetros, variables y umbral elegidos **solo** con validación cruzada agrupada en el 70 %.
- El 30 % se evaluó una sola vez con el modelo que eligió la validación cruzada.

In [ ]:
import pathlib, json, joblib
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, precision_recall_curve, ConfusionMatrixDisplay
ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == 'notebooks' else pathlib.Path.cwd()
R = ROOT / 'reports' / '05_modelado'
pd.set_option('display.width', 220); pd.set_option('display.max_columns', 40)
test = pd.read_csv(R / 'resultados_test_30.csv'); cv = pd.read_csv(R / 'resultados_cv_entrenamiento.csv')
pred = pd.read_csv(R / 'predicciones_test.csv'); part = pd.read_csv(R / 'particion_70_30.csv')
part.groupby(['particion', 'user_group']).agg(sesiones=('session_id', 'size'), participantes=('participant_id', 'nunique'))

## 1. Modelo seleccionado por heurística (30 % de prueba)

In [ ]:
cols = ['heuristica','modelo','variables','n','positivos','umbral','exactitud','exactitud_balanceada','precision',
        'sensibilidad','especificidad','f1','mcc','kappa','roc_auc','pr_auc','brier']
win = test[test.seleccionado_por_cv == True]
win[cols].round(3)

In [ ]:
win[['heuristica','modelo','f1_ic95','mcc_ic95','sensibilidad_ic95','especificidad_ic95','roc_auc_ic95','mcnemar_p_vs_base']]

In [ ]:
# ¿qué métricas superan 0.85?
m = ['exactitud','exactitud_balanceada','precision','sensibilidad','especificidad','f1','mcc','kappa','roc_auc','pr_auc']
(win.set_index('heuristica')[m] > 0.85).replace({True: '✔', False: ''})

## 2. Comparación de todos los modelos

In [ ]:
for h, g in test.groupby('heuristica'):
    print(f'=== {h}')
    display(g[['modelo','variables','f1','mcc','sensibilidad','especificidad','roc_auc','pr_auc','seleccionado_por_cv']].round(3).sort_values('mcc', ascending=False))

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4))
p = test[~test.modelo.str.startswith('Línea')].pivot(index='modelo', columns='heuristica', values='f1')
p.plot.barh(ax=ax); ax.axvline(0.85, ls='--', c='k', lw=1); ax.set_xlabel('F1 en prueba'); ax.set_title('F1 por modelo y heurística')
plt.tight_layout()

## 3. Validación cruzada en el 70 % (la base de la selección)

In [ ]:
cv[['heuristica','modelo','variables','hiperparametros','cv_roc_auc','cv_f1','cv_mcc','cv_sensibilidad','cv_especificidad']].round(3)

## 4. Curvas ROC y Precisión-Recall del modelo seleccionado

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(16, 7))
for i, (_, r) in enumerate(win.iterrows()):
    d = pred[(pred.heuristica == r.heuristica) & (pred.modelo == r.modelo)]
    fpr, tpr, _ = roc_curve(d.y, d.prob); pr, rc, _ = precision_recall_curve(d.y, d.prob)
    axes[0, i].plot(fpr, tpr); axes[0, i].plot([0, 1], [0, 1], 'k:'); axes[0, i].set_title(f'{r.heuristica} ROC (AUC={r.roc_auc:.2f})')
    axes[1, i].plot(rc, pr); axes[1, i].axhline(d.y.mean(), c='k', ls=':'); axes[1, i].set_title(f'{r.heuristica} PR (AP={r.pr_auc:.2f})')
plt.tight_layout()

## 5. Matrices de confusión

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(16, 3.8))
for ax, (_, r) in zip(axes, win.iterrows()):
    d = pred[(pred.heuristica == r.heuristica) & (pred.modelo == r.modelo)]
    ConfusionMatrixDisplay.from_predictions(d.y, (d.prob >= d.umbral).astype(int), ax=ax, colorbar=False,
                                            display_labels=['sin violación', 'violación'])
    ax.set_title(f'{r.heuristica} · {r.modelo}')
plt.tight_layout()

## 6. Importancia de variables (modelos de árboles seleccionados)

In [ ]:
for f in sorted((ROOT / 'models').glob('y_H*.joblib')):
    b = joblib.load(f); est = b['model'][-1]
    if hasattr(est, 'feature_importances_'):
        imp = pd.Series(est.feature_importances_, b['features']).sort_values(ascending=False).head(10)
    elif hasattr(est, 'estimators_') and hasattr(est.estimators_[0], 'feature_importances_'):
        imp = pd.Series(np.mean([e.feature_importances_ for e in est.estimators_], 0)[:len(b['features'])], b['features']).sort_values(ascending=False).head(10)
    else:
        print(f.name, '→ sin importancias nativas'); continue
    print(f'=== {f.name}'); display(imp.round(3).to_frame('importancia'))